In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from Functions import (laydowncoordinates, compute_A, energy, propose_move, valid_fold)

CONFIG = {
    # ---- problem setup ----
    "H": 1,
    "P": 0,
    "Gamma": [1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1],  # 20-bead sequence
    "target_E": -9,                    # published ground-state energy for this sequence

    # ---- schedule shape (must match what your SA benchmark table used) ----
    "num_sweeps": 10000,
    "beta_min": 0.1,
    "beta_max": 10.0,
    "num_stairs": 20,                  # step count for Linear/Geometric Staircase

    # ---- REINFORCE schedule handoff ----
    # path to the .pkl saved from the REINFORCE training notebook (has
    # "beta_schedule" and "best_beta_schedule" keys, one value per block)
    "reinforce_pkl_path": "reinforce_beta_schedules.pkl",
    "reinforce_key": "best_beta_schedule",   # or "beta_schedule" -- which one to include here

    # ---- convergence-tracking run ----
    "num_trace_trials": 200,           # trials per schedule -- smaller than your full 1000-trial
                                        # benchmark on purpose, this is a diagnostic figure, not
                                        # the headline number
    "record_every": 100,               # record energy every N sweeps

    # ---- output ----
    "outdir": "./convergence_results",
    "n_jobs": -1,                      # -1 = use all available cores
}




In [ ]:


def build_schedules(cfg):
    """Returns a dict of {name: beta_array}, every array length == num_sweeps.
    REINFORCE's per-block schedule is expanded (np.repeat) to the same
    per-sweep resolution as everything else, so downstream code doesn't need
    to treat it specially."""
    num_sweeps = cfg["num_sweeps"]
    beta_min, beta_max = cfg["beta_min"], cfg["beta_max"]
    num_stairs = cfg["num_stairs"]

    schedules = {
        "Linear": np.linspace(beta_min, beta_max, num_sweeps),
        "Geometric": beta_min * (beta_max / beta_min) ** (np.arange(num_sweeps) / (num_sweeps - 1)),
        "Logarithmic": beta_min + (beta_max - beta_min) * np.log1p(np.arange(num_sweeps)) / np.log1p(num_sweeps - 1),
        "Linear Staircase": np.repeat(np.linspace(beta_min, beta_max, num_stairs), num_sweeps // num_stairs),
        "Geometric Staircase": np.repeat(
            beta_min * (beta_max / beta_min) ** (np.arange(num_stairs) / (num_stairs - 1)),
            num_sweeps // num_stairs,
        ),
        "Quenching": np.full(num_sweeps, beta_max),
    }

    # sanity check: every classical schedule must be exactly num_sweeps long
    for name, arr in schedules.items():
        assert len(arr) == num_sweeps, f"{name} has length {len(arr)}, expected {num_sweeps}"

    # --- REINFORCE: load, expand to per-sweep resolution, append to the same dict ---
    reinforce_path = cfg["reinforce_pkl_path"]
    if os.path.exists(reinforce_path):
        with open(reinforce_path, "rb") as f:
            rl_state = pickle.load(f)
        rl_beta_raw = np.asarray(rl_state[cfg["reinforce_key"]])

        if len(rl_beta_raw) == num_sweeps:
            # Already a full per-sweep curve (e.g. the sigmoid notebook's saved
            # "beta_schedule" -- it's the expanded curve, not block values).
            schedules["REINFORCE"] = rl_beta_raw
            print(f"Loaded REINFORCE schedule from {reinforce_path} "
                  f"(already full-length: {num_sweeps} sweeps)")
        elif num_sweeps % len(rl_beta_raw) == 0:
            # Block-style schedule (e.g. the block/staircase REINFORCE notebook's
            # saved per-block array) -- expand each block to its hold length.
            num_blocks = len(rl_beta_raw)
            block_size = num_sweeps // num_blocks
            schedules["REINFORCE"] = np.repeat(rl_beta_raw, block_size)
            print(f"Loaded REINFORCE schedule from {reinforce_path} "
                  f"({num_blocks} blocks x {block_size} sweeps/block = {num_sweeps})")
        else:
            raise ValueError(
                f"REINFORCE schedule from {reinforce_path} has length {len(rl_beta_raw)}, "
                f"which neither equals num_sweeps={num_sweeps} nor evenly divides it. "
                f"Check that CONFIG['num_sweeps'] matches the num_time_steps used during "
                f"training for this schedule."
            )
        assert len(schedules["REINFORCE"]) == num_sweeps
    else:
        print(f"REINFORCE schedule not found at {reinforce_path} -- "
              f"proceeding with classical schedules only.")

    return schedules




def run_trial_tracked(Gamma, A, N, beta, H, record_every):
    """One trial under a fixed per-sweep beta array, using Barker acceptance
    (matching the dynamics REINFORCE's gradient was derived under). Returns
    an array of energies recorded every `record_every` sweeps."""
    while True:
        d = np.random.randint(0, 4, size=N - 1)
        coordinates = laydowncoordinates(d)
        if valid_fold(coordinates):
            break

    E = energy(coordinates, Gamma, A, H)
    num_records = len(beta) // record_every
    trace = np.empty(num_records)

    for i in range(len(beta)):
        while True:
            d_new = propose_move(d)
            coords_new = laydowncoordinates(d_new)
            if valid_fold(coords_new):
                break
        E_new = energy(coords_new, Gamma, A, H)
        deltaE = E_new - E

        p_accept = 1.0 / (1.0 + np.exp(deltaE * beta[i]))   # Barker
        if np.random.rand() < p_accept:
            d, coordinates, E = d_new, coords_new, E_new

        if (i + 1) % record_every == 0:
            trace[(i + 1) // record_every - 1] = E

    return trace


# ======================================================================
# Main
# ======================================================================

def main():
    cfg = CONFIG
    os.makedirs(cfg["outdir"], exist_ok=True)

    H = cfg["H"]
    Gamma = cfg["Gamma"]
    N = len(Gamma)
    A = compute_A(Gamma)

    schedules = build_schedules(cfg)
    print(f"\nSchedules to track: {list(schedules.keys())}\n")

    traces = {}
    for name, beta in schedules.items():
        print(f"Tracking {name} ({cfg['num_trace_trials']} trials)...")
        results = Parallel(n_jobs=cfg["n_jobs"], prefer="processes")(
            delayed(run_trial_tracked)(Gamma, A, N, beta, H, cfg["record_every"])
            for _ in range(cfg["num_trace_trials"])
        )
        traces[name] = np.array(results)   # shape (num_trace_trials, num_records)

    # ------------------------------------------------------------------
    # Save raw traces (for later re-plotting / bootstrapping without rerunning)
    # ------------------------------------------------------------------
    traces_path = os.path.join(cfg["outdir"], "convergence_traces.pkl")
    with open(traces_path, "wb") as f:
        pickle.dump({"traces": traces, "config": cfg}, f)
    print(f"\nSaved raw traces to {traces_path}")

    # ------------------------------------------------------------------
    # Plot: mean +/- std convergence curve, all schedules overlaid
    # ------------------------------------------------------------------
    x = np.arange(cfg["record_every"], cfg["num_sweeps"] + 1, cfg["record_every"])

    fig, ax = plt.subplots(figsize=(10, 6))
    for name, tr in traces.items():
        mean_curve = tr.mean(axis=0)
        std_curve = tr.std(axis=0)
        linewidth = 2.5 if name == "REINFORCE" else 1.5
        ax.plot(x, mean_curve, label=name, linewidth=linewidth)
        ax.fill_between(x, mean_curve - std_curve, mean_curve + std_curve, alpha=0.12)

    ax.set_xlabel("Sweep t")
    ax.set_ylabel("Energy (mean ± 1 std across trials)")
    ax.set_title("Convergence speed: energy vs. sweep, by schedule")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()

    fig_path = os.path.join(cfg["outdir"], "convergence_comparison.svg")
    plt.savefig(fig_path, bbox_inches="tight")
    plt.show()
    print(f"Saved {fig_path}")

    # ------------------------------------------------------------------
    # Summary table: sweep-to-threshold, a simple numeric convergence-speed metric
    # (first sweep at which the mean curve reaches within 10% of its own final value)
    # ------------------------------------------------------------------
    rows = []
    for name, tr in traces.items():
        mean_curve = tr.mean(axis=0)
        final_val = mean_curve[-1]
        threshold = final_val + 0.1 * abs(final_val)   # energies are negative; "within 10%" means less negative than this
        reached = np.where(mean_curve <= threshold)[0]
        sweep_to_threshold = x[reached[0]] if len(reached) > 0 else None
        rows.append({
            "Schedule": name,
            "Final Mean Energy (at num_sweeps)": final_val,
            "Sweep to reach 90% of final value": sweep_to_threshold,
        })
    summary = pd.DataFrame(rows).sort_values("Sweep to reach 90% of final value")
    summary_path = os.path.join(cfg["outdir"], "convergence_speed_summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"\nSaved {summary_path}\n")
    print(summary.to_string(index=False))

    print("\nDone.")


if __name__ == "__main__":
    main()